<h1 style="font-size: 3em; text-align: center;"><u>NLP - Sentiment Analysis</u></h1>

# Table of Contents

1. Introduction
2. Essential Object-Oriented Programming for Structuring Data
3. Data Loading
4. Data Preparation
5. Bag of Words Model
	1. Explanation of how bag of words models function
	2. Reasons for employing bag of words models
	3. Resssources and Articles
	4. Steps to Convert Text to Numerical Vectors using BOW
	5. Example Application
6. `CountVectorizer`
7. `TfidfVectorizer` - Understanding TF-IDF
8. Choosing the Right Model
	1. SVM - [Linear Support Vector Classification](https://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVC.html)
	2. [Decision Trees](https://scikit-learn.org/stable/modules/tree.html)
	3. [Naive Bayes](https://scikit-learn.org/stable/modules/naive_bayes.html) - GaussianNB
	4. [Logistic Regression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)
	5. Multi-layer Perceptron Classifier [(MLPClassifier)](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html)
9. Evaluation of the Models
	1. Mean Accuracy: Using the `model.score` Method
	2. Accuracy Classification Score: Using the `accuracy_score` Function
	3. Classification Report: Using the `classification_report` Function
	4. F1 Scores
10. Model Tuning Using Grid Search
11. Model Saving abd Loading

# 1. Introduction

**Sentiment Analysis** is a powerful tool in the field of **Natural Language processing (NLP)** that involves determining the emotional tone behind a body of text.  
It is widely used in various domains, such as marketing, customer service, and social media monitoring, to gain insights into public opinion and customer feedback.

In this notebook, we will explore the process of performing **Sentiment Analysis** on **fashion reviews**. Our goal is to classify these reviews as either positive or negative.  

By the end of this notebook, we will grasp the essential steps and techniques for structuring data for sentiment analysis, preparing and transforming text data, and selecting and evaluating various machine learning models.

# 2. Essential Object-Oriented Programming for Structuring Data

In [1]:
class Sentiment:
    NEGATIVE = "NEGATIVE"
    NEUTRAL = "NEUTRAL"
    POSITIVE = "POSITIVE"
    
class Review:
    def __init__(self, text, rating):
        self.text = text
        self.rating = rating
        self.sentiment = self.getSentiment()
        
    def getSentiment(self):
        if self.rating <= 2:
            return Sentiment.NEGATIVE
        elif self.rating == 3:
            return Sentiment.NEUTRAL
        else:
            return Sentiment.POSITIVE

In [2]:
import random
random.seed(0)

class ReviewCollection:
    def __init__(self, reviews):
        self.reviews = reviews
        
    def getText(self):
        return [x.text for x in self.reviews]
    
    def getSentiment(self):
        return [x.sentiment for x in self.reviews]
        
    def evenlyDistribute(self):
        negative = list(filter(lambda x: x.sentiment == Sentiment.NEGATIVE, self.reviews))
        positive = list(filter(lambda x: x.sentiment == Sentiment.POSITIVE, self.reviews))

        positive = positive[:len(negative)]
        self.reviews = negative + positive
        
        random.shuffle(self.reviews)

The `filter()` function is used to filter elements of an iterable (like a list) based on a function (predicate) that returns either `True` or `False` for each element.  
It creates an iterator of elements for which the function returns `True`.
```python
filter(function, iterable)
```
It returns an iterator, so you typically convert it to a list or iterate over it in a loop to access the filtered elements.


----
<div style="text-align: center;">
    <h3>Case Involving Neutrals</h3>
</div>

```python
	def evenlyDistribute(self):
        negative = list(filter(lambda x: x.sentiment == Sentiment.NEGATIVE, self.reviews))
        positive = list(filter(lambda x: x.sentiment == Sentiment.POSITIVE, self.reviews))
        neutral = list(filter(lambda x: x.sentiment == Sentiment.NEUTRAL, self.reviews))

        minCount = min(len(negative), len(positive), len(neutral))
        
        negative = negative[:minCount]
        positive = positive[:minCount]
        neutral = neutral[:minCount]
        
        self.reviews = negative + positive + neutral
        
        random.shuffle(self.reviews)
```

---

# 3. Data Loading

In [3]:
import json

dataset = 'Clothing_Shoes_Jewelry'
fashionPath = f'./LightData/Light_{dataset}.json'

fashionReviews = []
with open(fashionPath) as f:
    for line in f:
        review = json.loads(line)
        fashionReviews.append(Review(review['reviewText'], review['overall']))

---

If we were working on the [food dataset](./LightData/Food_Reviews/Light_Food_Reviews.json) instead of this [fashion dataset](./LightData/Light_Clothing_Shoes_Jewelry.json), we would need to take a different approach. The [food dataset](./LightData/Food_Reviews/Light_Food_Reviews.json) poses a challenge due to <u>special characters</u> in the original CSV file that are not properly encoded. We could handle this by either skipping lines with these special characters using exception handling as seen below, or by using <u>UTF-8 encoding</u> when loading the data and converting the CSV file to JSON, as demonstrated in [ConvertingData.py](./RawData/Food_Reviews/ConvertingData.py).

---

```python
dataset = 'Food_Reviews'
foodPath = f'./LightData/{dataset}/Light_{dataset}.json'
foodReviews = []
with open(foodPath, encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            review = json.loads(line)
            foodReviews.append(Review(review['review'], review['rating']))
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON on line: {line}\nError: {e}")
            print(f"Skipping invalid JSON line")
            continue
```

---

Loading data from the very large [Food_Reviews.json](./RawData/Food_Reviews/Food_Reviews.json) file is significantly slower due to its high computing power requirements.  
Therefore, we use [Light_Food_Reviews.json](./LightData/Food_Reviews/Light_Food_Reviews.json), a lighter and smaller version of the original file, for faster and more efficient processing.

---

# 4. Data Preparation

In [4]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(fashionReviews, train_size=2/3, random_state=42)

--- 

**Fun Fact:** The widespread use of 42 as a `random_state` by engineers is a playful homage to [The Hitchhiker's Guide to the Galaxy](https://www.imdb.com/title/tt0371724/) where it humorously represents the "Answer to the Ultimate Question of Life, the Universe, and Everything."

---

We are employing a **Bag-of-Words (BOW)** model for text classification. The dataset includes two key elements:

<div style="text-align:center;">
  <table border="1" style="margin-left:auto;margin-right:auto;">
    <tr>
      <td>X</td>
      <td>This denotes the textual data that serves as input for our BOW model.</td>
    </tr>
    <tr>
      <td>y</td>
      <td>This represents the categories or sentiments assigned to each text sample, which our model aims to predict.</td>
    </tr>
  </table>
</div>

Our goal is to train a model that effectively predicts the category or sentiment based on the provided textual data.

Typically, we prepare the X matrix and y vector for both training and testing sets as follows:
```python
				train_X = [x.text for x in train]
				train_y = [x.sentiment for x in train]
				test_X = [x.text for x in test]
				test_y = [x.sentiment for x in test]
```

However, random assignment of X and y can lead to an imbalance in the data, potentially causing the model to favor one category over others, thereby reducing overall performance. Specifically, our dataset contains more positive reviews than negative ones.

---

**Solution**: Ensuring a controlled distribution by creating the `ReviewCollection` class and the `evenlyDistribute()` function within it.

---

The reader of this notebook should know that this solution was not considered at first.  
Only when we encountered the poor performances of all the tested models did we realize that it was certainly a data issue, not a machine modeling issue.

---

In [5]:
trainCollection = ReviewCollection(train)
testCollection = ReviewCollection(test)

trainCollection.evenlyDistribute()
train_X = trainCollection.getText()
train_y = trainCollection.getSentiment()

testCollection.evenlyDistribute()
test_X = testCollection.getText()
test_y = testCollection.getSentiment()

print(train_y.count(Sentiment.POSITIVE))
print(train_y.count(Sentiment.NEGATIVE))

1023
1023


---

Clearly, we achieved a balanced distribution. We are now ensuring that our dataset includes as many positive reviews as negative ones.

---

# 5. Bag of Words Model

The **BoW model** is a foundational technique in **Natural Language Processing (NLP)** for text modeling.  
When applying algorithms in NLP, they typically operate on numerical data rather than raw text. Therefore, the Bag of Words model preprocesses text by converting it into a bag that counts the occurrences of the most frequently used words.

In this model, text data is represented as numerical feature vectors. Each feature corresponds to a word in the vocabulary, and the value in each feature represents either the frequency or presence of that word in the document.

## 5.1. Explanation of how bag of words models function

A vector space is a multi-dimensional space in which points are plotted. In a bag of words approach, each individual word becomes a separate dimension (or axis) of the vector space. If a text set has n number of words, the resulting vector space has n dimensions, one dimension for each unique word in the text set. The model then plots each separate text document as a point in the vector space.  
A point’s position along a certain dimension is determined by the number of times that dimension’s word appears within the point’s document.

For example, assume we have a text set in which the contents of two separate documents are respectively:

- Document 1: A rose is red, a violet is blue
- Document 2: My love is like a red, red rose

Because it is difficult to imagine anything beyond a three-dimensional space, we will limit ourselves to just that. A vector space for a corpus containing these two documents would have separate dimensions for red, rose, and violet.

Since red, rose, and violet all occur once in Document 1, the vector for that document in this space will be (1,1,1).  
In Document 2, red appears twice, rose once, and violet not at all. Thus, the vector point for Document 2 is (2,1,0).  
Both of these document-points will be mapped in the three-dimensional vector space as:
<div style="text-align:center">
    <img src="./Images/docs_1_2_3D.png" style="width:30%;">
</div>

Note that this figure visualizes text documents as data vectors in a three-dimensional feature space. But bag of words can also represent words as feature vectors in a data space. A feature vector signifies the value (occurrence) of a given feature (word) in a specific data point (document). So the feature vectors for red, rose, and violet in Documents 1 and 2 would look like:
<div style="text-align:center">
    <img src="./Images/docs_1_2_2D.png" style="width:30%;">
</div>

**Note**: that the <u>order of words in the original documents is irrelevant</u>.  
For a bag of words model, all that matters is each word’s number of occurrences across the text set.

## 5.2. Reasons for employing bag of words models

Text classification tasks interpret those words with high frequency in a document as representing the document’s main ideas.  
This is not an unreasonable assumption. For example, if some of the most frequent words in a document are president, voters, and election, there is a high probability the document is a political text, specifically discussing a presidential election.  
Text classification with bag of words then extrapolates that documents with similar content are similar in type.

## 5.3. Resources and Articles

- [IBM - What is bag of words?](https://www.ibm.com/topics/bag-of-words)
- [GeeksforGeeks - Bag of words (BoW) model in NLP](https://www.geeksforgeeks.org/bag-of-words-bow-model-in-nlp/)

## 5.4. Steps to Convert Text to Numerical Vectors using BOW


1. **Tokenization**: 
   - First, <u>tokenize the text data</u>, which involves <u>splitting it into individual words or terms</u>.  
   This step typically removes punctuation and splits the text based on whitespace.

   Example:
   ```python
   text = "This is an example sentence."
   tokens = text.split()
   ```
   Output:
   ```
   ['This', 'is', 'an', 'example', 'sentence.']
   ```

2. **Building the Vocabulary**: 
   - Create a vocabulary of all unique words present in your dataset.  
   Each word in the vocabulary will become a **feature** in your BOW representation.

   Example:
   ```python
   vocabulary = set(tokens)
   ```
   Output:
   ```
   {'This', 'is', 'an', 'example', 'sentence.'}
   ```

3. **Generating BOW Vectors**:
   - For each document or piece of text, <u>count the frequency of each word in the vocabulary</u> and <u>construct a numerical vector</u> where each element corresponds to the count of that word in the document.

   Example:
   ```python
   bowVector = []
   for word in vocabulary:
       bowVector.append(tokens.count(word))
   ```
   Output:
   ```
   [1, 1, 1, 1, 1]
   ```
   - In this example, each element in `bowVector` represents the count of corresponding words in the vocabulary within the text.

4. **Vectorization**:
   - Representing the text as a **bag** (multiset) of its words, disregarding grammar and word order but keeping multiplicity.


5. **Normalization (Optional)**:
   - <u>Normalize</u> the BOW vectors *if necessary* to account for varying document lengths **or** to improve model performance.

6. **Implementation Considerations**:
   - **Stopwords**: Consider removing common words (stopwords) like "the", "is", "and" from the vocabulary as they often don't contribute much to the meaning.
   - **Tokenization**: Use robust tokenization methods that handle punctuation, special characters, and other language-specific features appropriately.
   - **Vectorization Libraries**: Utilize libraries like scikit-learn in Python (`CountVectorizer` or `TfidfVectorizer`) or similar tools in other programming languages to efficiently perform BOW vectorization.


## 5.5. Example Application

Consider the text: "What is behind the table ?"

1. **Tokenization**: Split into tokens:
   - Tokens: ["What", "is", "behind", "the", "table", "?"]
   
2. **Normalization**: Convert tokens to lowercase (assuming we normalize the tokens):
   - Tokens: ["what", "is", "behind", "the", "table", "?"]
   
3. **Building a Vocabulary**: Create a vocabulary of unique words:
   - Vocabulary: {"what", "is", "behind", "the", "table", "?"}
   
4. **BOW Vectorization**: Count occurrences of each word in the vocabulary within the document:
   - Document: "What is behind the table?"
   - BOW Vector: [1, 1, 1, 1, 1, 1]
     - Explanation:
       - "what": 1 (appears once)
       - "is": 1 (appears once)
       - "behind": 1 (appears once)
       - "the": 1 (appears once)
       - "table": 1 (appears once)
       - "?": 1 (appears once)

Therefore, the Bag-of-Words vector representation for the text "What is behind the table?" is `[1, 1, 1, 1, 1, 1]`.  
Each element in the vector corresponds to the count of the respective word from the vocabulary in the document.

Each document will have a unique BOW vector based on the frequency of words in that document compared to the vocabulary.

<div style="text-align:center">
    <img src="./Images/BOW.png" alt="BOW">
</div>

# 6. `CountVectorizer` [(Documentation)](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html)

`CountVectorizer` is a utility in the `scikit-learn` (sklearn) library used for converting a <u>collection of text documents</u> into <u>a matrix of token counts</u>.  
It essentially transforms a <u>corpus of text</u> into a <u>sparse matrix</u> where each **row** represents a **document** and each **column** represents a **token** (word or n-gram).

Here’s how `CountVectorizer` works and its key features:

In [6]:
from sklearn.feature_extraction.text import CountVectorizer

In [7]:
corpus = [
    'This is the first document.',
    'This document is the second document.',
    'And this is the third one.',
    'Is this the first document?',
]

vectorizer = CountVectorizer()

vectorizer.fit(corpus)
X = vectorizer.transform(corpus)

**Note**: You can accomplish the same outcome with just this single line
`X = vectorizer.fit_transform(corpus)`

In [8]:
featureNames = vectorizer.get_feature_names_out()

print("Feature Names:")
print(featureNames)

print("\nDocument-Term Matrix:")
print(X.toarray())

Feature Names:
['and' 'document' 'first' 'is' 'one' 'second' 'the' 'third' 'this']

Document-Term Matrix:
[[0 1 1 1 0 0 1 0 1]
 [0 2 0 1 0 1 1 0 1]
 [1 0 0 1 1 0 1 1 1]
 [0 1 1 1 0 0 1 0 1]]


`X.toarray()` converts the **sparse matrix** `X` into a dense NumPy array for easier inspection.

Each row in this array represents a document from the original corpus, and each column represents a word from the vocabulary learned by `CountVectorizer`.  

The value at position `(i, j)` in the matrix represents the count of word `j` in document `i`.

# 7. `TfidfVectorizer` [(Documentation)](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html)

## Understanding TF-IDF

With standard BoW models, semantically irrelevant words (the, is, some, etc.) can have the highest term frequency, and so greatest weight in a model.  
<u>Term frequency-inverse document frequency</u> **(TF-IDF)** aims to correct this. While bag of words counts only the number of times a word appears in one document, TF-IDF accounts for the **word’s prevalence** throughout every document in a text set.

The more documents in which a given word appears, the greater TF-IDF reduces that word’s weight. In this way, TF-IDF is an example of <u>feature scaling</u> in ML Models.

Much like general BoW Models, NLP packages often have pre-exsiting functions for implementing TF-IDF, such as scikit-learn’s `TfidfVectorizer` function.

`TfidfVectorizer` in `scikit-learn` is similar to `CountVectorizer` but instead of using just the count of words, it computes the <u>Term Frequency-Inverse Document Frequency</u> **(TF-IDF)** value for each word in the document. TF-IDF reflects the importance of a word in a document relative to all documents in the corpus.

## Key Concepts

1. **Term Frequency (TF)**: Measures how frequently a term **t** (word) appears in a document **d**.
   $$TF(t, d) = \frac{\text{number of times  t appears in d}}{\text{total number of terms in d}}$$

2. **Inverse Document Frequency (IDF)**:
   - Measures how important a term is across the entire corpus.
   - Terms that occur frequently across many documents are penalized.
   $$IDF(t) = 1 + \log \left( \frac{N}{df(t)} \right)$$ 
   - Where N is the total number of documents in the document set, and $df(t)$ is the number of documents in the document set that contain the term t.

3. **TF-IDF**: Combines TF and IDF to calculate a weight for each term in each document.
$$\text{TF-IDF}(t, d)=\text{TF}(t, d) \times \text{IDF}(t)$$

## Usage of `TfidfVectorizer`:

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
  
corpus = [
      'This is the first document.',
      'This document is the second document.',
      'And this is the third one.',
      'Is this the first document?',
]
  
vectorizer = TfidfVectorizer()

X = vectorizer.fit_transform(corpus)

featureNames = vectorizer.get_feature_names_out()

print("Feature Names:")
print(featureNames)

print("\nDocument-Term Matrix:")
print(X.toarray())

Feature Names:
['and' 'document' 'first' 'is' 'one' 'second' 'the' 'third' 'this']

Document-Term Matrix:
[[0.         0.46979139 0.58028582 0.38408524 0.         0.
  0.38408524 0.         0.38408524]
 [0.         0.6876236  0.         0.28108867 0.         0.53864762
  0.28108867 0.         0.28108867]
 [0.51184851 0.         0.         0.26710379 0.51184851 0.
  0.26710379 0.51184851 0.26710379]
 [0.         0.46979139 0.58028582 0.38408524 0.         0.
  0.38408524 0.         0.38408524]]


The resulting `X` is a **sparse** matrix where each row represents a document and each column represents a word from the vocabulary, weighted by TF-IDF.

## Returning to our data

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
train_X_vectors = vectorizer.fit_transform(train_X)
test_X_vectors = vectorizer.transform(test_X)
type(test_X_vectors)

scipy.sparse._csr.csr_matrix

```python
   test_X_vectors = vectorizer.transform(test_X)
```
This method transforms `test_X` into a TF-IDF encoded matrix (`test_X_vectors`)

**Important Notes**:
- After transformation, `train_X_vectors` and `test_X_vectors` will typically be sparse matrices (`scipy.sparse.csr_matrix`), which are efficient for handling large datasets with many features.

- Make sure that `train_X` and `test_X` are properly formatted as lists or arrays of strings (documents) before using `fit_transform()` and `transform()` respectively.

- Adjusting parameters in `TfidfVectorizer` such as `max_features`, `max_df`, `min_df`, can also impact the feature extraction process.

In [11]:
print(train_X[0])
print(train_X_vectors[0].toarray())

I have very... round breasts, but I'm a D so I need support.  I had the same &#34;pointy&#34; experience - if I loosen the straps I get no support.  If I loosen it too much I go from pointy, to saggy material.  The edge has this itchy looping to it, the rings are relatively flimsy and all together it looks uncomfortable and poorly made. It is really uncomfortable on me, not to say it wouldn't work for some people.  I think that this bra fits some people that need some &#34;uplifting&#34; but those that may already have it, well, we might as well start vogue-ing.  That's what I did.  Got a great laugh out of my husband, then I put it back in the box, and returned it.  If I wore it for a day there is no doubt in my mind I would be knocking things off counters and running into stuff with my &#34;torpedoes&#34;.  No amount of camisole, t-shirt, or any other outer covering could hide this shape - felt like if I had a vintage dress this bra would have shaped me perfectly, albeit uncomfortabl

# 8. Choosing the Right Model

```python
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
train_X_vectors = vectorizer.fit_transform(train_X)
test_X_vectors = vectorizer.transform(test_X)
```
Once this preprocessing step is set up, we can proceed to select an appropriate model.

## 8.1. SVM - [Linear Support Vector Classification](https://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVC.html)

In [12]:
from sklearn.svm import SVC

lsvc = SVC(kernel='linear', random_state=42)

lsvc.fit(train_X_vectors, train_y)

print(test_X[7])
print(lsvc.predict(test_X_vectors[7])[0])

This ring is just perfect! The design is just as shown, and the setting, while large, is not overbearing. I love it!
POSITIVE


## 8.2. [Decision Trees](https://scikit-learn.org/stable/modules/tree.html)

In [13]:
from sklearn.tree import DecisionTreeClassifier

dtc = DecisionTreeClassifier()

dtc.fit(train_X_vectors, train_y)

print(test_X[10])
print(dtc.predict(test_X_vectors[11])[0])

I don't know who thought this was cute but it isn't. I am extremely offended by the title of this wig. Since when did my natural textured hair become ghetto?? Amazon, you messed up when you decided to carry this wig on your site and California Costumes for making it! You would never find a wig titled "Trailer Trash Wig" so why would you give this wig a title that personifies a stereo type?I am disgusted and deeply offended by this and I will submitting a formal complaint about this to corporate!
NEGATIVE


## 8.3. [Naive Bayes](https://scikit-learn.org/stable/modules/naive_bayes.html) - GaussianNB

In [14]:
from sklearn.naive_bayes import GaussianNB

gnbc = GaussianNB()

train_X_dense = train_X_vectors.toarray()

gnbc.fit(train_X_dense, train_y)

print(test_X[12])
print(gnbc.predict(test_X_vectors[12].toarray())[0])

THE PANTS ARE REALLY CUTE THE BACK POCKET DESING ARE A PLUS BUT, THEY RUN PRETTY SMALL TO BE A PLUS SIZE PANTS, I ALREADY RETURNED THEM.
POSITIVE


The `GaussianNB` classifier from scikit-learn requires dense data.  
Sparse data is typically used when we have high-dimensional data with many zeros, optimized for memory efficiency.

To resolve this issue, we can convert our sparse data to a dense numpy array using the `.toarray()` method.

By calling `.toarray()` on our `train_X_vectors` (a sparse matrix, from `TfidfVectorizer`), we convert it into a dense numpy array that `GaussianNB` can work with.

## 8.4. [Logistic Regression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)

In [15]:
from sklearn.linear_model import LogisticRegression

lrc = LogisticRegression()

lrc.fit(train_X_vectors, train_y)

print(test_X[19])
print(lrc.predict(test_X_vectors[19])[0])

I just received these, and they are gorgeous!  The colors are beautiful together, and they sparkle like crazy.  They are very comfortable on the ear, and they are sterling and leverback!! I can't wait to wear them. Thank you for making these beautifulSwarovski Crystal earrings!!
POSITIVE


## 8.5. Multi-layer Perceptron Classifier [(MLPClassifier)](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html)

In [16]:
from sklearn.neural_network import MLPClassifier

mlpc = MLPClassifier(max_iter=250, random_state=42)
mlpc.fit(train_X_vectors, train_y)

print(test_X[13])
print(mlpc.predict(test_X_vectors[13])[0])

This dress fits me perfect! It hugs my bottom nicely. It is comfortable. I got a lot of compliments on it...
POSITIVE


# 9. Evaluation of the Models

## 9.1. Mean Accuracy: Using the `model.score` Method

This method takes `test_X_vectors` and `test_y` as arguments.

In [17]:
print("Linear Support Vector Classifier (LSVC) Accuracy:", lsvc.score(test_X_vectors, test_y))
print("Logistic Regression Classifier (LRC) Accuracy:", lrc.score(test_X_vectors, test_y))
print("MLP Classifier (MLPC) Accuracy:", mlpc.score(test_X_vectors, test_y))
print("Decision Tree Classifier (DTC) Accuracy:", dtc.score(test_X_vectors, test_y))
print("Gaussian Naive Bayes Classifier (GNB) Accuracy:", gnbc.score(test_X_vectors.toarray(), test_y))

Linear Support Vector Classifier (LSVC) Accuracy: 0.8563714902807775
Logistic Regression Classifier (LRC) Accuracy: 0.8585313174946004
MLP Classifier (MLPC) Accuracy: 0.8358531317494601
Decision Tree Classifier (DTC) Accuracy: 0.6965442764578834
Gaussian Naive Bayes Classifier (GNB) Accuracy: 0.6263498920086393


## 9.2. Accuracy Classification Score: Using the `accuracy_score` Function

This function takes `y_pred` and `test_y` as arguments.

In [18]:
from sklearn.metrics import accuracy_score

classifiers = {
    'Linear Support Vector Classifier': lsvc,
    'Logistic Regression Classifier': lrc,
    'MLP Classifier': mlpc,
    'Decision Tree Classifier': dtc,
    'Gaussian Naive Bayes Classifier': gnbc
}

for clf_name, clf in classifiers.items():
    if clf_name == 'Gaussian Naive Bayes Classifier':
        y_pred = clf.predict(test_X_vectors.toarray())
    else:
        y_pred = clf.predict(test_X_vectors)

    accuracy = accuracy_score(test_y, y_pred)
    print(f'{clf_name} Accuracy Score: {accuracy}')

Linear Support Vector Classifier Accuracy Score: 0.8563714902807775
Logistic Regression Classifier Accuracy Score: 0.8585313174946004
MLP Classifier Accuracy Score: 0.8358531317494601
Decision Tree Classifier Accuracy Score: 0.6965442764578834
Gaussian Naive Bayes Classifier Accuracy Score: 0.6263498920086393


## 9.3. Classification Report: Using the `classification_report` Function

In [19]:
from sklearn.metrics import classification_report

y_pred = lsvc.predict(test_X_vectors)
print(classification_report(test_y, y_pred))

              precision    recall  f1-score   support

    NEGATIVE       0.85      0.87      0.86       463
    POSITIVE       0.86      0.85      0.85       463

    accuracy                           0.86       926
   macro avg       0.86      0.86      0.86       926
weighted avg       0.86      0.86      0.86       926



## 9.4. F1 Scores

In [20]:
from sklearn.metrics import f1_score
import pandas as pd

y_pred = lsvc.predict(test_X_vectors)
labels=[Sentiment.POSITIVE, Sentiment.NEGATIVE]
f1_scores = f1_score(test_y, y_pred, average=None, labels=labels)


results = pd.DataFrame({'Category': labels, 'F1 Score': f1_scores})

print(results.to_string(index=False))

Category  F1 Score
POSITIVE  0.854962
NEGATIVE  0.857754


# 10. Model Tuning Using Grid Search

In [21]:
from sklearn.model_selection import GridSearchCV

parameters = {'kernel': ('linear', 'rbf'), 'C': (1, 1.5, 2, 4, 8, 16, 32)}
svc = SVC()
clf = GridSearchCV(svc, parameters, cv=5)
clf.fit(train_X_vectors, train_y)

GridSearchCV(cv=5, estimator=SVC(),
             param_grid={'C': (1, 1.5, 2, 4, 8, 16, 32),
                         'kernel': ('linear', 'rbf')})

In [22]:
print("Best parameters found: ", clf.best_params_)

print("Best cross-validation score: {:.2f}".format(clf.best_score_))

Best parameters found:  {'C': 1, 'kernel': 'linear'}
Best cross-validation score: 0.84


In [23]:
clf.score(test_X_vectors, test_y)

0.8563714902807775

# 11. Model Saving and Loading

In [24]:
import pickle

## Saving the Model

In [25]:
with open('./SavedModels/SentimentClassifier.pkl', 'wb') as f:
    pickle.dump(clf, f)

## Loading the Model

In [26]:
with open('./SavedModels/SentimentClassifier.pkl', 'rb') as f:
    SentimentClassifier = pickle.load(f)

In [27]:
print(test_X[22])
print(SentimentClassifier.predict(test_X_vectors[22])[0])

The young lady who received this watch as a gift is enjoying the watch. It was the correct type and color for her age.
POSITIVE
